# Demo — Least Privilege IAM Policy Comparison

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/bedrock-companion/blob/main/day1/demos/demo-least-privilege-iam/demo-least-privilege-iam.ipynb)

This notebook is a follow-along demo.

Day 1 — Block 3: Securing the Agent (Identity and Least Privilege)

Compares a dangerous, broad IAM policy against a production-ready
least-privilege policy. Demonstrates how to use conditions to restrict
tools by resource, tag, and gateway.

In [ ]:
# Install required dependencies
!pip install boto3 --quiet

In [ ]:
import json

def print_section(title: str):
    print(f"\n{'=' * 60}")
    print(f"  {title}")
    print(f"{'=' * 60}")

def show_bad_policy():
    """A common anti-pattern: over-permissioned agent role."""
    print_section("ANTI-PATTERN: Broad IAM Policy")

    bad_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": "lambda:InvokeFunction",
                "Resource": "*",
            }
        ],
    }
    print(f"  {json.dumps(bad_policy, indent=2)}")
    print("\n  Why is this dangerous?")
    print("  - The agent can invoke ANY Lambda function in the account.")
    print("  - If prompted maliciously, it could trigger a data deletion function.")
    print("  - Fails the principle of least privilege.")

def show_good_policy():
    """Production pattern: scoped by resource and gateway."""
    print_section("PRODUCTION PATTERN: Least Privilege")

    good_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "AllowSpecificTools",
                "Effect": "Allow",
                "Action": "bedrock-agentcore-gateway:InvokeTool",
                "Resource": "arn:aws:bedrock-agentcore:...:tool/get_order_status",
                "Condition": {
                    "StringEquals": {
                        "aws:ResourceTag/Environment": "Prod",
                    }
                },
            }
        ],
    }
    print(f"  {json.dumps(good_policy, indent=2)}")
    print("\n  Why is this better?")
    print("  - Binds the agent strictly to Gateway tools (not raw Lambda/S3).")
    print("  - Scopes down to a specific tool ARN.")
    print("  - Requires the target to be tagged Environment=Prod.")

def show_abac_policy():
    """Attribute-Based Access Control (ABAC) using runtimeUserId."""
    print_section("ADVANCED: Attribute-Based Access Control (ABAC)")

    abac_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "AllowUserSpecificData",
                "Effect": "Allow",
                "Action": "bedrock-agentcore-gateway:InvokeTool",
                "Resource": "arn:aws:bedrock-agentcore:...:tool/get_user_profile",
                "Condition": {
                    "StringEquals": {
                        "bedrock-agentcore:RuntimeUserId": "${aws:PrincipalTag/Department}"
                    }
                },
            }
        ],
    }
    print(f"  {json.dumps(abac_policy, indent=2)}")
    print("\n  How it works:")
    print("  - The policy dynamically compares the caller (RuntimeUserId)")
    print("  - To the resource tags or principal tags.")
    print("  - Eliminates the need to write unique policies for every user.")

def main():
    print("Least Privilege IAM — Instructor Demo\n")
    show_bad_policy()
    show_good_policy()
    show_abac_policy()

    print_section("Key Takeaways")
    print("  1. Wildcards (*) in agent IAM roles are production blockers.")
    print("  2. Agents should only invoke tools via AgentCore Gateway.")
    print("  3. Use tags and context keys (ABAC) to scale security dynamically.")

if __name__ == "__main__":
    main()
